# 04 — Model Comparison: TF-IDF vs BERT vs Hybrid

This notebook justifies the final scoring formula used in `app/ai/hybrid_scorer.py`.

**What we explore:**
- Side-by-side comparison of TF-IDF, BERT, and Hybrid scores
- Case where TF-IDF alone fails (paraphrased CV)
- Case where BERT alone fails (keyword-stuffed CV)
- Radar chart comparison per candidate
- Weight sensitivity analysis (what happens if we change the 32/48/20 split)

**Production formula (in `hybrid_scorer.py`):**
```python
TFIDF_WEIGHT = 0.32
BERT_WEIGHT  = 0.48
QUIZ_WEIGHT  = 0.20
hybrid = QUIZ_WEIGHT * quiz_score + TFIDF_WEIGHT * tfidf + BERT_WEIGHT * bert
```

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from math import pi

from app.ai.tfidf_model import compute_tfidf_score
from app.ai.bert_model import compute_bert_score
from app.ai.hybrid_scorer import score_resume, TFIDF_WEIGHT, BERT_WEIGHT, QUIZ_WEIGHT

print(f'Production weights — TF-IDF: {TFIDF_WEIGHT}, BERT: {BERT_WEIGHT}, Quiz: {QUIZ_WEIGHT}')

## 1. Test Profiles

In [ ]:
JOB_DESC = """
Backend Software Engineer — Python, Flask, REST API, MongoDB, Docker, AWS, JWT, WebSocket, scikit-learn, spaCy.
Must have CI/CD experience with GitHub Actions or Jenkins. NLP library knowledge is a strong plus.
"""

profiles = {
    'Strong Match': (
        """
        John Smith — Backend Engineer. 5 years Python Flask REST APIs. MongoDB, Docker, AWS.
        JWT auth, WebSocket. scikit-learn, spaCy. CI/CD Jenkins. NLP pipeline work.
        """,
        0.9   # quiz score (90%)
    ),
    'Semantic Only (Paraphrased)': (
        """
        Alice Tan — API Developer. Server-side web framework development.
        NoSQL document storage. Cloud container deployments. Token-based access control.
        Real-time data streaming. Machine learning library usage for text analysis.
        """,
        0.7   # quiz: 70%
    ),
    'Keyword Stuffed': (
        """
        Bob Garcia — Python Flask MongoDB Docker AWS JWT WebSocket scikit-learn spaCy CI/CD Jenkins REST API NLP.
        (No actual experience. Just listed every keyword from the job description.)
        """,
        0.1   # quiz: 10% — reveals lack of real knowledge
    ),
    'Partial Match': (
        """
        Jane Doe — Full Stack Dev. Python Django REST. PostgreSQL. React. GitHub Actions CI/CD.
        No MongoDB, Docker, or NLP experience.
        """,
        0.5
    ),
    'Unrelated': (
        """
        Tom Brown — Graphic Designer. Photoshop, Illustrator, InDesign, typography, brand identity.
        Motion graphics, social media content. Zero programming experience.
        """,
        0.0
    ),
}

print('Profiles defined.')

## 2. Score All Profiles

In [ ]:
rows = []
for name, (cv_text, quiz_pct) in profiles.items():
    result = score_resume(JOB_DESC, cv_text, quiz_score=quiz_pct)
    rows.append({
        'Profile': name,
        'Quiz %': round(quiz_pct * 100, 1),
        'TF-IDF %': round(result.tfidf_score * 100, 1),
        'BERT %': round(result.bert_score * 100, 1),
        'Hybrid %': round(result.hybrid_score * 100, 1),
        'Skill Match %': round(result.skill_match_pct, 1),
        'RF Prediction': result.ml_prediction,
        'RF Probability %': round(result.ml_probability, 1)
    })

df = pd.DataFrame(rows)
print(df[['Profile','Quiz %','TF-IDF %','BERT %','Hybrid %','RF Prediction']].to_string(index=False))

## 3. Bar Chart — All Scores Side by Side

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

x = np.arange(len(rows))
width = 0.2
metrics = ['TF-IDF %', 'BERT %', 'Quiz %', 'Hybrid %']
colors = ['#4f86c6', '#f4845f', '#f4d06f', '#67b99a']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    vals = [r[metric] for r in rows]
    ax.bar(x + (i - 1.5) * width, vals, width, label=metric, color=color)

ax.set_ylabel('Score (%)')
ax.set_title('TF-IDF vs BERT vs Quiz vs Hybrid Score')
ax.set_xticks(x)
ax.set_xticklabels([r['Profile'] for r in rows], rotation=15, ha='right', fontsize=9)
ax.legend()
ax.set_ylim(0, 115)
plt.tight_layout()
plt.show()

## 4. Key Cases — Where Each Model Succeeds or Fails

In [ ]:
case_df = df[df['Profile'].isin(['Semantic Only (Paraphrased)', 'Keyword Stuffed'])][['Profile','TF-IDF %','BERT %','Quiz %','Hybrid %']]
print(case_df.to_string(index=False))
print()
print('Semantic Only: TF-IDF misses it (no exact keywords), but BERT catches it.')
print('Keyword Stuffed: TF-IDF and BERT both score OK, but Quiz (10%) pulls Hybrid way down.')
print('=> Quiz score is the fraud-detection layer that pure NLP cannot provide.')

## 5. Radar Chart per Candidate

In [ ]:
categories = ['TF-IDF', 'BERT', 'Quiz', 'Skill Match']
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, axes = plt.subplots(1, len(rows), figsize=(4 * len(rows), 4), subplot_kw=dict(polar=True))
if len(rows) == 1:
    axes = [axes]

colors = ['#4f86c6', '#f4845f', '#e63946', '#457b9d', '#2a9d8f']

for ax, r, color in zip(axes, rows, colors):
    values = [
        r['TF-IDF %'],
        r['BERT %'],
        r['Quiz %'],
        r['Skill Match %']
    ]
    values += values[:1]

    ax.plot(angles, values, color=color, linewidth=2)
    ax.fill(angles, values, color=color, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=8)
    ax.set_ylim(0, 100)
    ax.set_title(r['Profile'], size=9, pad=12)

plt.suptitle('Radar Chart — Score Profile per Candidate', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 6. Weight Sensitivity Analysis

In [ ]:
# Strong Match candidate — how does the hybrid score change as we shift weights?
cv_text, quiz_pct = profiles['Strong Match']
result_raw = score_resume(JOB_DESC, cv_text, quiz_score=quiz_pct)
t = result_raw.tfidf_score
b = result_raw.bert_score
q = quiz_pct

weight_combos = [
    ('Equal (33/33/33)', 0.33, 0.33, 0.33),
    ('TF-IDF heavy (60/30/10)', 0.60, 0.30, 0.10),
    ('BERT heavy (20/70/10)', 0.20, 0.70, 0.10),
    ('No Quiz (40/60/0)', 0.40, 0.60, 0.00),
    ('Production (32/48/20)', 0.32, 0.48, 0.20),
]

print(f'Base scores — TF-IDF: {t:.3f}, BERT: {b:.3f}, Quiz: {q:.3f}\n')
print(f'{"Config":<30} {"Hybrid %":>10}')
print('-' * 42)
for label, wt, wb, wq in weight_combos:
    h = wt * t + wb * b + wq * q
    marker = ' ← production' if 'Production' in label else ''
    print(f'{label:<30} {h*100:>9.1f}%{marker}')

## Summary

| Scenario | TF-IDF | BERT | Quiz | Hybrid outcome |
|----------|--------|------|------|----------------|
| Perfect match | ✅ High | ✅ High | ✅ High | 🏆 Top rank |
| Paraphrased CV | ❌ Low | ✅ High | ✅ OK | ✅ Still ranks well |
| Keyword stuffed | ✅ High | ✅ High | ❌ Low | ⚠️ Penalised |
| Partial match | OK | OK | OK | Middle rank |
| Unrelated | ❌ Low | ❌ Low | ❌ Zero | ❌ Bottom |

**Conclusion:** The 32/48/20 split provides the best balance — BERT dominates for semantic quality, TF-IDF anchors keyword precision, and Quiz acts as the knowledge-verification fraud filter.